In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

# Day 4 Lab: Sequential Logic Fundamentals

> **Week 1, Session 4** · Accelerated HDL for Digital System Design · UCF ECE

## Overview

| | |
|---|---|
| **Duration** | ~2 hours |
| **Prerequisites** | Pre-class video (50 min): clocks, edges, nonblocking assignment, flip-flops, resets, counters |
| **Deliverable** | LED blinker at ~1 Hz + counter value on 7-segment display |
| **Tools** | Icarus Verilog + GTKWave (simulation), Yosys + nextpnr (synthesis) |

## Learning Objectives

| SLO | Description |
|-----|-------------|
| 4.1 | Write `always @(posedge clk)` blocks with synchronous reset |
| 4.2 | Use nonblocking assignment (`<=`) correctly in sequential blocks |
| 4.3 | Implement D flip-flops with enable and reset |
| 4.4 | Design counter-based clock dividers |
| 4.5 | Debug sequential logic using Icarus Verilog simulation and GTKWave |
| 4.6 | Integrate sequential and combinational modules into a working system |

## Exercises

### Exercise 1: D Flip-Flop — Simulate First! (25 min)
Implement in `starter/w1d4_ex1_d_ff.v`, run testbench with `make ex1_sim`. Open waveforms in GTKWave. Mark the moment `i_d` changes vs. when `o_q` changes.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex1_d_ff.v
// =============================================================================
// Exercise 1: D Flip-Flop
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Part A: Basic D-FF with synchronous reset
// Part B: D-FF with clock enable (d_ff_en — uncomment below)
// =============================================================================

module d_ff (
    input  wire i_clk,
    input  wire i_reset,
    input  wire i_d,
    output reg  o_q
);

    // TODO: Implement the D flip-flop
    // On posedge i_clk:
    //   if i_reset: o_q <= 0
    //   else: o_q <= i_d
    //
    // always @(posedge i_clk) begin
    //     ...
    // end

endmodule

// Part B: D-FF with clock enable — uncomment and implement
//
// module d_ff_en (
//     input  wire i_clk,
//     input  wire i_reset,
//     input  wire i_enable,
//     input  wire i_d,
//     output reg  o_q
// );
//
//     // On posedge i_clk:
//     //   if reset: clear
//     //   else if enable: capture i_d
//     //   else: hold (no latch — this is sequential!)
//
//     always @(posedge i_clk) begin
//         // TODO
//     end
//
// endmodule

In [ ]:
%%writefile ex1_tb_d_ff.v
// =============================================================================
// Exercise 1: D Flip-Flop Testbench
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Verify: D is captured on posedge clk, reset clears output.
// Use GTKWave to mark the moment i_d changes vs. the moment o_q changes.
// =============================================================================

`timescale 1ns / 1ps

module tb_d_ff;

    reg  clk, reset, d;
    wire q;

    d_ff uut (
        .i_clk(clk), .i_reset(reset), .i_d(d), .o_q(q)
    );

    // 10 ns clock period (100 MHz for easy reading)
    always #5 clk = ~clk;

    integer pass_count = 0;
    integer fail_count = 0;

    task check;
        input expected;
        input [8*30-1:0] name;
    begin
        if (q === expected) begin
            $display("PASS: %0s — q=%b (expected %b)", name, q, expected);
            pass_count = pass_count + 1;
        end else begin
            $display("FAIL: %0s — q=%b (expected %b)", name, q, expected);
            fail_count = fail_count + 1;
        end
    end
    endtask

    initial begin
        $dumpfile("build/ex1.vcd");
        $dumpvars(0, tb_d_ff);

        // Initialize
        clk = 0; reset = 1; d = 0;

        // Reset
        @(posedge clk); #1;
        check(0, "After reset, q=0");

        // Release reset, set d=1
        reset = 0; d = 1;
        @(posedge clk); #1;
        check(1, "d=1 captured");

        // d=0
        d = 0;
        @(posedge clk); #1;
        check(0, "d=0 captured");

        // Toggle d rapidly
        d = 1;
        @(posedge clk); #1;
        check(1, "d=1 captured again");

        // Reset while d=1
        reset = 1;
        @(posedge clk); #1;
        check(0, "Reset overrides d=1");

        // Summary
        $display("");
        $display("========================================");
        $display("D Flip-Flop: %0d passed, %0d failed", pass_count, fail_count);
        $display("========================================");

        #20 $finish;
    end

endmodule

### Exercise 2: Loadable Register (20 min)
4-bit register with load enable. `make ex2_sim` to verify load/hold/reset behavior.

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex2_register_4bit.v
// =============================================================================
// Exercise 2: 4-Bit Loadable Register
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module register_4bit (
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_load,
    input  wire [3:0] i_data,
    output reg  [3:0] o_q
);

    // TODO: Implement the register
    // On posedge clk:
    //   if reset: clear to 0
    //   else if load: capture i_data
    //   else: hold current value

endmodule

In [ ]:
%%writefile ex2_tb_register.v
// =============================================================================
// Exercise 2: Register Testbench
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

`timescale 1ns / 1ps

module tb_register;

    reg        clk, reset, load;
    reg  [3:0] data;
    wire [3:0] q;

    register_4bit uut (
        .i_clk(clk), .i_reset(reset), .i_load(load),
        .i_data(data), .o_q(q)
    );

    always #5 clk = ~clk;

    integer pass_count = 0;
    integer fail_count = 0;

    task check;
        input [3:0] expected;
        input [8*40-1:0] name;
    begin
        if (q === expected) begin
            $display("PASS: %0s — q=%h (expected %h)", name, q, expected);
            pass_count = pass_count + 1;
        end else begin
            $display("FAIL: %0s — q=%h (expected %h)", name, q, expected);
            fail_count = fail_count + 1;
        end
    end
    endtask

    initial begin
        $dumpfile("build/ex2.vcd");
        $dumpvars(0, tb_register);

        clk = 0; reset = 1; load = 0; data = 4'h0;

        // TODO: Add test sequence:
        // 1. Reset the register — verify o_q == 0
        // 2. Load 4'hA — verify o_q == A
        // 3. Deassert load, present new data — verify hold
        // 4. Load 4'h5 — verify it changes
        // 5. Reset again — verify o_q == 0

        @(posedge clk); #1;
        check(4'h0, "After reset");

        // Release reset, load A
        reset = 0; load = 1; data = 4'hA;
        @(posedge clk); #1;
        check(4'hA, "Loaded 0xA");

        // Hold test
        load = 0; data = 4'hF;
        @(posedge clk); #1;
        check(4'hA, "Hold — should still be 0xA");

        // Load 5
        load = 1; data = 4'h5;
        @(posedge clk); #1;
        check(4'h5, "Loaded 0x5");

        // Reset
        reset = 1;
        @(posedge clk); #1;
        check(4'h0, "Reset clears to 0");

        $display("");
        $display("========================================");
        $display("Register: %0d passed, %0d failed", pass_count, fail_count);
        $display("========================================");

        #20 $finish;
    end

endmodule

### Exercise 3: LED Blinker (25 min) ★ KEY EXERCISE
Free-running counter with multi-speed LED output. `make ex3` to program. LED1 slowest, LED4 fastest — visually demonstrates binary counting.

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex3_led_blinker.v
// =============================================================================
// Exercise 3: Free-Running Counter + LED Blinker
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// The Go Board has a 25 MHz clock. To blink an LED at ~1 Hz:
//   25,000,000 / 2 = 12,500,000 cycles per half-period
//   Need a counter that counts to 12,500,000 - 1, then toggles
//
// Part A: Single LED blink at ~1 Hz
// Part B: Multi-speed — different LEDs from different counter bits
// =============================================================================

module led_blinker (
    input  wire i_clk,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // TODO: Create a 24-bit counter
    // reg [23:0] r_counter = 24'd0;
    //
    // always @(posedge i_clk) begin
    //     r_counter <= r_counter + 24'd1;
    // end

    // TODO: Toggle an LED register at terminal count
    // Or simpler: just use counter bits directly!
    //
    // Part B (multi-speed):
    //   o_led1 <- ~r_counter[23]  (~1.5 Hz)
    //   o_led2 <- ~r_counter[22]  (~3 Hz)
    //   o_led3 <- ~r_counter[21]  (~6 Hz)
    //   o_led4 <- ~r_counter[20]  (~12 Hz)

    // Placeholder — replace with your implementation
    assign o_led1 = 1'b1;
    assign o_led2 = 1'b1;
    assign o_led3 = 1'b1;
    assign o_led4 = 1'b1;

endmodule

### Exercise 4: 7-Segment Counter — Week 1 Capstone (30 min) ★ CAPSTONE
Running hex counter on the display. Integrates clock division + counting + combinational decoding. `make ex4`.

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex4_seg_counter.v
// =============================================================================
// Exercise 4: 7-Segment Counter (Week 1 Capstone)
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Display a running hex counter (0-F) on the 7-segment display.
// Count rate: slow enough to read (~2 Hz)
//
// This exercise integrates:
//   - Sequential logic (counter)
//   - Combinational logic (7-seg decoder)
//   - Clock division (tick generation)
// =============================================================================

module seg_counter (
    input  wire i_clk,
    output wire o_led1,        // heartbeat (fast blink)
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    // ---- Clock Divider: generate a ~2 Hz tick ----
    // TODO: Count to 12,500,000 - 1, then assert a one-cycle tick
    //
    // reg [23:0] r_clk_count = 24'd0;
    // reg        r_tick      = 1'b0;
    //
    // always @(posedge i_clk) begin
    //     if (r_clk_count == 24'd12_499_999) begin
    //         r_clk_count <= 24'd0;
    //         r_tick      <= 1'b1;
    //     end else begin
    //         r_clk_count <= r_clk_count + 24'd1;
    //         r_tick      <= 1'b0;
    //     end
    // end

    // ---- 4-bit Display Counter ----
    // TODO: Increment on each tick (0 → F → 0 → ...)
    //
    // reg [3:0] r_display = 4'd0;
    //
    // always @(posedge i_clk) begin
    //     if (r_tick)
    //         r_display <= r_display + 4'd1;
    // end

    // ---- 7-Segment Decoder ----
    // TODO: Decode r_display to segment pattern
    // You can inline the case statement or instantiate hex_to_7seg

    // ---- Heartbeat ----
    // TODO: Blink LED1 from a counter bit for visual feedback

    // Placeholder outputs — replace with your implementation
    assign o_led1 = 1'b1;
    assign o_segment1_a = 1'b1;
    assign o_segment1_b = 1'b1;
    assign o_segment1_c = 1'b1;
    assign o_segment1_d = 1'b1;
    assign o_segment1_e = 1'b1;
    assign o_segment1_f = 1'b1;
    assign o_segment1_g = 1'b1;

endmodule

### Exercise 5: Dual-Speed Blinker (15 min)
Two independent dividers, complementary LED pairs. `make ex5`.

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex5_dual_blinker.v
// =============================================================================
// Exercise 5: Dual-Speed Blinker
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Two independent clock dividers driving two LED pairs at different rates.
// =============================================================================

module dual_blinker (
    input  wire i_clk,
    output wire o_led1,    // ~1 Hz
    output wire o_led2,    // ~1 Hz (inverted)
    output wire o_led3,    // ~4 Hz
    output wire o_led4     // ~4 Hz (inverted)
);

    // TODO: Counter 1 — ~1 Hz (count to ~12.5M)
    // TODO: Counter 2 — ~4 Hz (count to ~3.125M)
    // LED pairs show complementary patterns (one on, one off)

    assign o_led1 = 1'b1;  // TODO
    assign o_led2 = 1'b1;  // TODO
    assign o_led3 = 1'b1;  // TODO
    assign o_led4 = 1'b1;  // TODO

endmodule

### Exercise 6 — Stretch: Up/Down Counter (if time permits)
Button-controlled counter on 7-seg. Will be bouncy without debouncing — this previews Day 5!

#### 📝 Exercise 6 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex6_updown_counter.v
// =============================================================================
// Exercise 6 (Stretch): Up/Down Counter with 7-Seg Display
// Day 4 · Sequential Logic Fundamentals
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Button-controlled counter displayed on 7-segment.
// sw1 = count up, sw2 = count down (raw, undebounced — will be bouncy!)
// This previews the need for debouncing (Week 2, Day 5).
// =============================================================================

module updown_counter (
    input  wire i_clk,
    input  wire i_switch1,   // count up (raw, active-low)
    input  wire i_switch2,   // count down (raw, active-low)
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    // TODO:
    // 1. Simple edge detection on buttons (will be imperfect without debounce)
    // 2. 4-bit up/down counter
    // 3. 7-seg decoder
    //
    // Note: Without debouncing, the counter will sometimes skip values.
    // This is expected and motivates Day 5's debounce module!

endmodule

## Deliverable Checklist

- [ ] Exercise 1: Testbench passes, waveforms examined in GTKWave
- [ ] Exercise 2: Testbench passes all 5 test cases
- [ ] Exercise 3: LEDs blink at visible rates on board
- [ ] Exercise 4: 7-seg counts 0→F at readable speed ← **primary deliverable**
- [ ] At minimum: Exercise 3 (LED blinker) working on board

## Quick Reference

In [ ]:
!make ex1_sim      # Simulate D flip-flop

In [ ]:
!make ex2_sim      # Simulate register

In [ ]:
!make ex3          # Program LED blinker

In [ ]:
!make ex4          # Program 7-seg counter (capstone)

In [ ]:
!make ex5          # Program dual blinker

In [ ]:
!make ex6          # Program up/down counter (stretch)

In [ ]:
!make clean

## End of Week 1! 🎉

You went from zero HDL to a counter running on a display in 4 days. That's a real accomplishment. Next week: debouncing, testbench methodology, FSMs, and parameterization.